# Application of Radial Equilibrium Equation for a Rotor - Inverse Problem
## Adaptation of the code RE-DES by Lewis (Turbomachines Performance Analysis)

The radial equilibrium equation can be reordened and written as

$$\boxed{
\frac{\text{d}}{\text{d}r} c_x(r)^2 = 2\left( \omega - \frac{c_\theta(r)}{r}\right)
\frac{\text{d} (rc_\theta(r))}{\text{d}r}
}  \tag{1}
$$

that gives the solution
$$
c_x(r) = \sqrt{f(r) + k} \tag{2}
$$
where
$$
f(r) = 2 \int_{r_h}^r \left( \omega - \frac{c_\theta(r)}{r}\right)\text{d} (rc_\theta(r)) \tag{3}
$$

The aim is, given all the data: $Q$, $\omega$, $r_h$, $r_t$ and the function
$c_\theta(r)$, compute $c_x(r)$ and the angle $\beta_2(r)$ that fullfils the
required flowrate.

The necessary modules are imported

In [9]:
import numpy as np
import pandas as pd
pd.set_option('display.precision', 2)
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

The data for the fan is input in this cell.
The hub to tip ratio and the rms radius, $r_\text{rms} = \sqrt{\frac{r_h^2+r_t^2}{2}}$ are computed, and the swirl distribution is also chosen.
For numerical computations, the $r$ domain is divided in $m$ intervals, and $n$ for output of results. Velocity at $r_{rms}$, $c_{\theta,\text{rms}}=c_\theta(r=r_\text{rms})$, is calculated from Euler's equation, and $c_x$ in $r_\text{rms}$ is as well estimated, assuming that it is the average velocity given by the flow rate, $\overline{c_x}$.

In [10]:
rho = 1.2                           # Density of air, kg/m³
Dh = 0.064                          # Diameter of hub, m
rh = Dh/2                           # Radius of hub, m
Dt = 0.25                           # Diameter of tip, m
rt = Dt/2                           # Radius of tip, m
h = rh/rt                           # Hub-tip ratio
rrms = np.sqrt(0.5*(rh*rh+rt*rt))   # RMS radius, m
n = 10                              # number of outputs
m = 601                             # number of interpolation points
r = np.linspace(rh,rt,m)            # Discretization of the radius for interpolation, m
Qdata = 1716                        # Flow rate, m³/h
Qdata = Qdata/3600                  # Flow rate, m³/s
omega = 2275                        # Rotational speed, rpm
omega = omega*np.pi/30              # Rotational speed, rad/s
Delta_p0_target = 70                # Pressure rise, Pa

ctrms = Delta_p0_target/(rho*omega*rrms)         # c_theta,rms, m/s
cxm = Qdata/(np.pi*(rt*rt-rh*rh))   # c_x,rms, m/s
print("Flow rate = {:0.4f} m³/s".format(Qdata))
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))


Flow rate = 0.4767 m³/s
Hub to tip ratio = 0.2560
rms = 0.0912 m
ctheta_rms = 2.6837 m/s
cx_rms = 10.3916 m/s


- **Free Vortex**:
  $$ c_\theta = \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
- **Forced Vortex**:
  $$ c_\theta = Ar $$
  where 
  $$ A = \frac{c_{\theta,rms}}{r_{rms}} $$
- **Constant Vortex**:
  $$ c_\theta = c_{\theta,rms} $$
- **Mixed Vortex**:
  $$ c_\theta = A(r-r_{rms}) + \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
  and
  $$ A = \frac{\Delta c_\theta}{r_t - r_h} + \frac{B}{r_t r_h} $$
  where $\Delta c_\theta$ is the variability of $c_\theta$ between the tip and the hub, that is, some kind of "strength" of the vortex
- **Arbitray Vortex**:
  The user can define any function $c_\theta(r)$


In [11]:
# User choice: "free", "forced", "constant", "mixed", "arbitrary", "table"
flow_type = "table"

if flow_type == "free":
    B = ctrms*rrms
    ctheta = B/r
    print("Free vortex flow")
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "forced":
    A = ctrms/(rrms)
    ctheta = A*r
    print("Forced vortex flow")
    print("A = {:.4f} 1/s".format(A))
elif flow_type == "constant":
    ctheta = ctrms*np.ones(r.size)
    print("Constant ctheta flow")
    print("ctheta = {:.4f} m/s".format(ctrms))
elif flow_type == "mixed":
    delta_ctheta = 1
    B = ctrms*rrms
    A = delta_ctheta/(rt-rh) + B/(rt*rh)
    ctheta = A*(r-rrms) + B/r
    print("Mixed vortex flow")
    print("A = {:.4f} 1/s".format(A))
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "arbitrary":
    ctheta = 2*omega/3*r-9.08/np.sqrt(r)
elif flow_type == "table":
    # For example, values obtained from a reference design, CFD, or experimental extraction
    c_theta_array = np.array([0.90278909, 1.28974431, 1.66424824, 2.03205183, 2.39580352,
       2.75690324, 3.11616483, 3.47409568, 3.83102958, 4.1871957 ])
    r_stations = np.linspace(rh, rt, len(c_theta_array))
    ctheta_spline = CubicSpline(r_stations, c_theta_array)
    ctheta = ctheta_spline(r)

And now, the function $f(r)$ is computed by numerical integration (Eq. (3))

In [12]:
f = 2.0 * cumulative_trapezoid(omega-np.divide(ctheta,r),
                         np.multiply(r,ctheta),initial=0)

### First approximation

The first aproximation of the value of $k$ is with the assumption that
$c_{x,\text{rms}} = \overline{c_x}$

In [13]:
frms = CubicSpline(r,f)(rrms)
k = cxm*cxm-frms
print("First approximation of k: \n k = {:.4f} m²/s²".format(k))

First approximation of k: 
 k = 6.2765 m²/s²


Values of $DF < 0.6$ and a first estimation of $C_D$ are defined. With this assumptions and data, ${C_L}$, ${σ}$ (solidity) and ${c_x}$ along the entire length of the profile are calculated.

With these data, the chord of the profile along the length of the blade is calculated, thus defining the geometry of the blade.


$$
σ = \frac {cos(β_1) (tan(β_1) - tan(β_2))}{2 D_F - 2  [1 - \frac{cos(β_1)}{cos(β_2) } ] } \tag{4}
$$

$$
C_L = \frac {2  cos(β_m)  (tan(β_1) - tan(β_2))}{σ} - C_Dtan(β_m) \tag{5}
$$

The contribution of Samuel Limonchi (course 2023-24 of MUREM) to this part of the notebook is acknowledged.

In [14]:
DF_target = 0.4
CD = 0.01
Nblades = 5
def solve_fan(k):
    cx = np.sqrt(np.maximum(k + f, 1.0e-6))
    Q = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    Delta_p0 = rho * omega * r * ctheta
    Delta_p0_avg = 2*trapezoid(np.multiply(r,Delta_p0),r)/(rt*rt-rh*rh)
    data_list = []
    for i in range(n):
        rdata = rh + (rt-rh)*i/(n-1)
        cthetadata = float(CubicSpline(r,ctheta)(rdata)) # That is in order to treat it as a float and not an array
        cxans = float(CubicSpline(r,cx)(rdata)) # That is in order to treat it as a float and not an array
        alpha2 = np.rad2deg(np.arctan(cthetadata/cxans))
        beta2 = np.rad2deg(np.arctan((omega*rdata-cthetadata)/cxans))
        beta1 = np.rad2deg(np.arctan(omega*rdata/cxans))
        cosbeta1 = np.cos(np.deg2rad(beta1))
        cosbeta2 = np.cos(np.deg2rad(beta2))
        tanbeta1 = np.tan(np.deg2rad(beta1))
        tanbeta2 = np.tan(np.deg2rad(beta2))
        tanbetam = 0.5*(tanbeta1 + tanbeta2)
        beta_m = np.rad2deg(np.arctan(tanbetam))
        cosbetam = 1 / np.sqrt(1 + tanbetam*tanbetam)
        solidity = (cosbeta1* (tanbeta1 - tanbeta2)) / (2*DF_target - 2 * (1 - (cosbeta1/cosbeta2) ))
        bladespace = 2*np.pi*rdata/Nblades
        chord =  solidity * bladespace
        CL = (2/solidity) * (cosbetam * (tanbeta1 - tanbeta2)) - CD*tanbetam
        Delta_p0_data = rho * omega * rdata * cthetadata
        data_list.append({
            "Radius (m)": rdata,
            "c_theta (m/s)": cthetadata,
            "c_x (m/s)": cxans,
            "alpha_2 (deg)": alpha2,
            "beta_1 (deg)": beta1,
            "beta_2 (deg)": beta2,
            "beta_m (deg)": beta_m,
            "Solidity": solidity,
            "CL": CL,
            "Chord (mm)": chord * 1000,
            "Delta p0 (Pa)": Delta_p0_data
        })
    print("Q = {:.3f} m^3/s".format(Q))
    errorQ = np.abs(Qdata-Q)/Qdata * 100
    print("error in flow rate = {:.1e} %".format(errorQ))
    print("Delta p0 avg = {:.2f} Pa".format(Delta_p0_avg))
    errorP0 = np.abs(Delta_p0_target-Delta_p0_avg)/Delta_p0_target * 100
    print("error in pressure rise = {:.1e} %".format(errorP0))
    df = pd.DataFrame(data_list)
    df = df.round({"Chord (mm)":0})
    return df

df = solve_fan(k)

Q = 0.454 m^3/s
error in flow rate = 4.7e+00 %
Delta p0 avg = 78.82 Pa
error in pressure rise = 1.3e+01 %


In [15]:
df

,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Delta p0 (Pa)
0,0.03,0.90,2.51,19.82,71.81,69.56,70.75,0.19,1.21,8.0,8.26
1,0.04,1.29,4.12,17.37,67.76,64.88,66.40,0.20,1.21,11.0,15.61
2,0.05,1.66,5.54,16.72,66.17,63.02,64.69,0.21,1.21,14.0,25.06
3,0.06,2.03,6.88,16.45,65.37,62.06,63.81,0.21,1.21,17.0,36.60
4,0.07,2.40,8.18,16.32,64.90,61.50,63.30,0.21,1.20,20.0,50.23
5,0.08,2.76,9.46,16.24,64.60,61.15,62.98,0.22,1.20,23.0,65.94
6,0.09,3.12,10.73,16.20,64.40,60.90,62.76,0.22,1.20,26.0,83.74
7,0.10,3.47,11.98,16.17,64.26,60.73,62.60,0.22,1.20,29.0,103.62
8,0.11,3.83,13.23,16.15,64.16,60.60,62.48,0.22,1.20,32.0,125.59
9,0.12,4.19,14.48,16.13,64.08,60.51,62.40,0.22,1.20,34.0,149.63


### More precise computation


Instead of estimating $k$ with the assumption of $c_x$ in $r_{rms}$ being the average value, a more accurate computation
can be performed forcing the flow rate to be the input one (equation (5.47) and figure 5.6)

In [23]:
from scipy.optimize import brentq

def QFunction(k):
    cx = np.sqrt(np.maximum(k + f, 1.0e-6))
    Q_temptative = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    return Qdata-Q_temptative

k = brentq(QFunction,min(0.5*k,5*k),max(0.5*k,5*k))
print("k = {:.4f} m²/s²".format(k))

k = 15.0563 m²/s²


In [24]:
df = solve_fan(k)
df

Q = 0.477 m^3/s
error in flow rate = 1.2e-14 %
Delta p0 avg = 78.82 Pa
error in pressure rise = 1.3e+01 %


,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Delta p0 (Pa)
0,0.03,0.90,3.88,13.10,63.02,60.0,61.59,0.17,1.27,7.0,8.26
1,0.04,1.29,5.08,14.25,63.27,60.0,61.72,0.19,1.24,10.0,15.61
2,0.05,1.66,6.28,14.84,63.40,60.0,61.79,0.20,1.23,13.0,25.06
3,0.06,2.03,7.49,15.17,63.47,60.0,61.83,0.21,1.22,16.0,36.60
4,0.07,2.40,8.70,15.39,63.52,60.0,61.86,0.21,1.22,19.0,50.23
5,0.08,2.76,9.92,15.54,63.55,60.0,61.88,0.21,1.21,22.0,65.94
6,0.09,3.12,11.13,15.64,63.57,60.0,61.89,0.21,1.21,25.0,83.74
7,0.10,3.47,12.34,15.72,63.59,60.0,61.90,0.22,1.21,28.0,103.62
8,0.11,3.83,13.56,15.78,63.60,60.0,61.91,0.22,1.21,31.0,125.59
9,0.12,4.19,14.78,15.82,63.61,60.0,61.91,0.22,1.21,34.0,149.63
